# Etapa 06 — MLflow e persistência skops

Política `mlflow_skops_v1`. Implementada e validada em fixtures; publicação da
execução real pelo autor ainda pendente. [Runbook](../../docs/MLFLOW.md).
Não há treinamento nem avaliação do teste nesta etapa. Os outputs ficam vazios
até execução local; não reproduzimos resultados de fixtures como evidência real.


## Decisão

Preservar o baseline real, publicar três runs principais independentes e usar
skops com assinatura e pipeline completo. Cada run tem um modelo, seu scaler
quando necessário, métricas e identidade da Gold. A migração lê somente a validação
para conferir scores. Joblib original é carregado apenas com confiança explícita;
recargas posteriores usam skops. Não há autolog, registry ou promoção.


## Publicação no terminal

```bash
poetry install
make validate
BASELINE_PATH="data/processed/handbook/6e67dbd0a3bfe0d7ec33abc4bce5f37cd4ff0d6a/baseline_v1/f6d7aca720f74316b4183f97b6d866ab"
poetry run python -m fraud_detection_mlops.modeling.tracking publish "$BASELINE_PATH" --trust-local-models
poetry run python -m fraud_detection_mlops.modeling.tracking verify "$BASELINE_PATH"
make mlflow-ui
```

A flag confirma os próprios modelos locais; não use a migração para artefatos
joblib de origem desconhecida. O [runbook](../../docs/MLFLOW.md) explica ambiente,
retomada e backup. UI: <http://127.0.0.1:5001>.


In [1]:
from fraud_detection_mlops.bronze import DEFAULT_INVENTORY, load_inventory
from fraud_detection_mlops.modeling.tracking import publish_baseline
from fraud_detection_mlops.modeling.train import DEFAULT_BASELINE

BASELINE_RUN_ID = "f6d7aca720f74316b4183f97b6d866ab"
source_commit = load_inventory(DEFAULT_INVENTORY)["source"]["commit"]
baseline_path = DEFAULT_BASELINE / source_commit / "baseline_v1" / BASELINE_RUN_ID
tracked = publish_baseline(baseline_path, verify_only=True)
print(tracked)

2026-10-07 21:09:57.627 | INFO     | fraud_detection_mlops.config:<module>:11 - PROJ_ROOT path is: /home/wanderson/Documentos/fraud-detection-mlops
/home/wanderson/Documentos/fraud-detection-mlops/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

verified: dummy_prior


verified: logistic_regression


verified: hist_gradient_boosting
{'schema_version': 1, 'version': 'mlflow_skops_v1', 'baseline_run_id': 'f6d7aca720f74316b4183f97b6d866ab', 'baseline_manifest_sha256': '31e1ad433a5df9c42ee17b8ecfa9e8659e815d0d82b84ad39c10a483f10309e7', 'tracking_uri': 'sqlite:////home/wanderson/Documentos/fraud-detection-mlops/data/tracking/mlflow.db', 'experiment_id': '1', 'models': [{'model_id': 'dummy_prior', 'mlflow_run_id': '112aed5b405b4dbc90296f65eeb1fbcb', 'model_uri': 'runs:/112aed5b405b4dbc90296f65eeb1fbcb/model', 'reused': True}, {'model_id': 'logistic_regression', 'mlflow_run_id': '5c71bf8c647e4c878c21d057ae613103', 'model_uri': 'runs:/5c71bf8c647e4c878c21d057ae613103/model', 'reused': True}, {'model_id': 'hist_gradient_boosting', 'mlflow_run_id': 'd3be86000fd24dc8a053dbcab778d4b6', 'model_uri': 'runs:/d3be86000fd24dc8a053dbcab778d4b6/model', 'reused': True}], 'test_evaluated': False, 'refit': False, 'environment': {'mlflow': '3.17.0', 'skops': '0.16.0', 'scikit-learn': '1.9.1', 'numpy': '2

In [2]:
from mlflow import MlflowClient
import pandas as pd

client = MlflowClient(tracking_uri=tracked["tracking_uri"])
rows = []
for item in tracked["models"]:
    run = client.get_run(item["mlflow_run_id"])
    rows.append(
        {
            "model_id": item["model_id"],
            "run_id": run.info.run_id,
            "status": run.info.status,
            **run.data.metrics,
        }
    )
print(pd.DataFrame(rows).to_string(index=False))

              model_id                           run_id   status  validation_rows  validation_fraud_count  validation_fraud_rate  validation_average_precision  validation_roc_auc  validation_daily_customer_precision_at_100  fit_seconds  predict_seconds
           dummy_prior 112aed5b405b4dbc90296f65eeb1fbcb FINISHED          67255.0                   580.0               0.008624                      0.008624            0.500000                                    0.021429     0.034777         0.000788
   logistic_regression 5c71bf8c647e4c878c21d057ae613103 FINISHED          67255.0                   580.0               0.008624                      0.435001            0.873168                                    0.480000     0.343365         0.005431
hist_gradient_boosting d3be86000fd24dc8a053dbcab778d4b6 FINISHED          67255.0                   580.0               0.008624                      0.623949            0.884953                                    0.540000     0.602763      

## Critérios e limites

Três runs `FINISHED`, sem parent run; AP reconciliada com a execução de origem;
classes `[0, 1]`, assinatura com 19 features e scores equivalentes após recarga.
Repetição reutiliza runs compatíveis; falhas não são promovidas a sucesso. O teste
permanece reservado. Tempos registrados são os históricos de fit/predict.

O próximo treinamento deverá produzir skops desde a origem em um novo contrato.
Diagnóstico, hipóteses estatísticas, otimização e quality gate serão implementados
antes de promover modelos. Ainda não há serving, monitoramento ou CT operacional.
